# VIDEO-DETECT(VD):     关于yolo的训练部署代码



> 下载python环境（conda）

1、安装Anaconda、并新建虚拟环境（python=3.8）、切换清华镜像源pip

2、pytorch安装

3、安装ultralytics（yolov8）

https://github.com/ultralytics/ultralytics/ 下载

解压
cd ultralytics-main

pip install -e.


### 模型训练

In [ ]:
from ultralytics import YOLO

a = YOLO('runs/detect/train8/weights/last.pt')

a.train(
    data='yolov8-data.yaml',   #数据集配置文件
    epochs=10,          #训练轮次
    batch=16,           #每次训练的批量
    device='cpu'        #gpu=0 cpu='cpu'
    # imgsz=640,
    # lr0=0.01,
    # device=0,
    # workers=4,
    # optimizer="AdamW",
    # hsv_h=0.015,
    # fliplr=0.5,
)


### 运用模型

In [ ]:
from ultralytics import YOLO

a1 = YOLO('runs/detect/train5/weights/best.pt')

a1("C:/Users/00103933/Desktop/111.mp4", show=True, save=True)


### 损失函数改进

In [ ]:
from ultralytics import YOLO
import numpy as np

# 自定义损失函数类
class AdaptiveWeightedLoss:
    def __init__(self):
        self.original_loss = None  # 原始YOLO损失函数
        
    def otsu_threshold(self, feature_map):
        """使用Otsu算法计算阈值"""
        # 将特征图转换为灰度概率
        hist, _ = np.histogram(feature_map.cpu().detach().numpy().flatten(), bins=256, range=(0, 1))
        prob = hist / hist.sum() 
        
        # 计算类概率和类内平均灰度
        cumsum = np.cumsum(prob)
        cummean = np.cumsum(prob * np.arange(256))
        
        # 最大化类间方差
        variance = np.zeros(256)
        for t in range(1, 256):
            w0 = cumsum[t-1]
            w1 = 1 - w0
            if w0 == 0 or w1 == 0:
                continue
                
            mean0 = cummean[t-1] / w0
            mean1 = (cummean[255] - cummean[t-1]) / w1
            variance[t] = w0 * w1 * (mean0 - mean1) ** 2 
            
        threshold = np.argmax(variance)
        return threshold / 255.0
        
    def calculate_adaptive_weight(self, feature_map, box):
        """计算自适应权重"""
        # 使用Otsu算法分割前景和背景
        threshold = self.otsu_threshold(feature_map)
        binary_map = (feature_map > threshold).float()
        
        # 计算前景像素数量
        star = binary_map.sum().item()
        
        # 计算标注框内的像素总数
        sbox = box.prod().item()
        
        # 计算自适应权重
        if sbox > 0:
            w = 1 - (star / sbox)
        else:
            w = 1.0
            
        return w
        
    def __call__(self, pred, target):
        """改进的损失函数"""
        # 获取原始损失
        original_loss = self.original_loss(pred, target)
        
        # 分解损失为定位损失、置信度损失和分类损失
        lbox = original_loss['box']  # 定位损失
        lobj = original_loss['obj']  # 置信度损失
        lcls = original_loss['cls']  # 分类损失
        
        # 计算自适应权重
        feature_map = pred[0]  # 假设第一个输出是特征图
        box = target[0]['boxes']  # 假设第一个目标是边界框
        
        w = self.calculate_adaptive_weight(feature_map, box)
        
        # 应用自适应权重到定位损失和置信度损失
        weighted_loss = {
            'box': w * lbox,
            'obj': w * lobj,
            'cls': lcls  # 分类损失保持不变
        }
        
        return weighted_loss

# 创建YOLO模型
a = YOLO('runs/detect/train8/weights/last.pt')

# 设置自定义损失函数
adaptive_loss = AdaptiveWeightedLoss()
adaptive_loss.original_loss = a.model.loss_fn
a.model.loss_fn = adaptive_loss

# 训练模型
a.train(
    data='yolov8-data.yaml',   # 数据集配置文件
    epochs=10,                  # 训练轮次
    batch=16,                   # 每次训练的批量,平衡计算资源和训练效率
    device='cpu',              # gpu=0 cpu='cpu'
    imgsz=640,                 # 输入图像尺寸640×640,平衡特征信息和计算开销
    hsv_h=0.015,              # 色调调整参数,模拟不同环境光照变化
    fliplr=0.5,               # 随机水平翻转概率,扩展数据多样性
    workers=4,                 # 数据加载的工作进程数
    optimizer="AdamW",         # 优化器选择
    dynamic_resize=True,       # 启用动态输入尺度调整
)